In [1]:
# =========================================================
# PS6E4 Ensemble Notebook
#
# Load npy from:
#   /kaggle/input/datasets/nuwaaaa0000/ps06e04-oof-ensemble
#   /kaggle/input/datasets/nuwaaaa0000/ps06e04-test-ensemble
#   /kaggle/input/datasets/nuwaaaa0000/ps06e04-oof
#   /kaggle/input/datasets/nuwaaaa0000/ps06e04-test
#
# Models:
#   S = xgb_sudhanshu_seed42_low_more_0p030 LB 0.98111
#   D = xgb_orig_domain_multiseed_tunedavg LB 0.98052
#   A = ensemble_2repro_xgbote LB 0.98037
#
# All npy class order:
#   [High, Low, Medium]
# =========================================================

import os
import gc
import numpy as np
import pandas as pd

from sklearn.metrics import balanced_accuracy_score, recall_score

pd.set_option("display.max_columns", None)

# =========================================================
# 0. PATHS / CONFIG
# =========================================================
TARGET = "Irrigation_Need"

TRAIN_PATH = "/kaggle/input/competitions/playground-series-s6e4/train.csv"
TEST_PATH  = "/kaggle/input/competitions/playground-series-s6e4/test.csv"

OOF_ENSEMBLE_DIR  = "/kaggle/input/datasets/nuwaaaa0000/ps06e04-oof-ensemble"
TEST_ENSEMBLE_DIR = "/kaggle/input/datasets/nuwaaaa0000/ps06e04-test-ensemble"

OOF_DIR  = "/kaggle/input/datasets/nuwaaaa0000/ps06e04-oof"
TEST_DIR = "/kaggle/input/datasets/nuwaaaa0000/ps06e04-test"

STD_LABELS = ["High", "Low", "Medium"]

label_to_std = {
    "High": 0,
    "Low": 1,
    "Medium": 2,
}

print("OOF_ENSEMBLE_DIR :", OOF_ENSEMBLE_DIR)
print("TEST_ENSEMBLE_DIR:", TEST_ENSEMBLE_DIR)
print("OOF_DIR          :", OOF_DIR)
print("TEST_DIR         :", TEST_DIR)

# =========================================================
# 1. LOAD TRAIN / TEST
# =========================================================
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

y_std = train[TARGET].map(label_to_std).values
test_ids = test["id"].values

print("\nData shapes:")
print("train:", train.shape)
print("test :", test.shape)
print("y_std:", y_std.shape)

# =========================================================
# 2. LIST NPY FILES
# =========================================================
def list_npy_files(directory):
    print("\n" + "=" * 90)
    print("DIR:", directory)
    print("=" * 90)

    if not os.path.exists(directory):
        print("NOT FOUND")
        return []

    files = sorted([f for f in os.listdir(directory) if f.endswith(".npy")])
    for f in files:
        print(f)
    return files

oof_ens_files = list_npy_files(OOF_ENSEMBLE_DIR)
test_ens_files = list_npy_files(TEST_ENSEMBLE_DIR)
oof_files = list_npy_files(OOF_DIR)
test_files = list_npy_files(TEST_DIR)

# =========================================================
# 3. FILE LOADER
# =========================================================
def find_oof_file(filename):
    search_dirs = [
        OOF_DIR,
        OOF_ENSEMBLE_DIR,
        "/kaggle/working",
    ]

    for d in search_dirs:
        path = os.path.join(d, filename)
        if os.path.exists(path):
            return path

    raise FileNotFoundError(
        f"OOF file not found: {filename}\n"
        "探した場所:\n"
        + "\n".join(search_dirs)
    )

def find_test_file(filename):
    search_dirs = [
        TEST_DIR,
        TEST_ENSEMBLE_DIR,
        "/kaggle/working",
    ]

    for d in search_dirs:
        path = os.path.join(d, filename)
        if os.path.exists(path):
            return path

    raise FileNotFoundError(
        f"TEST file not found: {filename}\n"
        "探した場所:\n"
        + "\n".join(search_dirs)
    )

def normalize_probs(p):
    p = np.asarray(p, dtype=np.float64)
    p = np.clip(p, 1e-15, None)
    p = p / p.sum(axis=1, keepdims=True)
    return p

def load_oof(filename):
    path = find_oof_file(filename)
    arr = np.load(path).astype(np.float64)
    arr = normalize_probs(arr)

    print("\nLoaded OOF")
    print("file :", filename)
    print("path :", path)
    print("shape:", arr.shape)

    return arr

def load_test(filename):
    path = find_test_file(filename)
    arr = np.load(path).astype(np.float64)
    arr = normalize_probs(arr)

    print("\nLoaded TEST")
    print("file :", filename)
    print("path :", path)
    print("shape:", arr.shape)

    return arr

# =========================================================
# 4. MODEL FILES
# =========================================================
FILES = {
    # S: current best single model
    "S_xgb_seed42_lowmore_0.98111": {
        "oof":  "oof_xgb_sudhanshu_seed42_low_more_0p030_std_0.98111.npy",
        "test": "test_xgb_sudhanshu_seed42_low_more_0p030_std_0.98111.npy",
        "lb": 0.98111,
    },

    # D: previous XGB original-domain multiseed tunedavg
    "D_xgb_orig_domain_tunedavg_0.98052": {
        "oof":  "oof_xgb_orig_domain_multiseed_tunedavg_std_0.98052.npy",
        "test": "test_xgb_orig_domain_multiseed_tunedavg_std_0.98052.npy",
        "lb": 0.98052,
    },

    # A: previous self ensemble
    "A_2repro_xgbote_0.98037": {
        "oof":  "oof_ensemble_2repro_xgbote_0.98037.npy",
        "test": "test_ensemble_2repro_xgbote_0.98037.npy",
        "lb": 0.98037,
    },
}

models = {}

for name, f in FILES.items():
    oof = load_oof(f["oof"])
    tes = load_test(f["test"])

    assert oof.shape == (len(train), 3), (name, "OOF", oof.shape)
    assert tes.shape == (len(test), 3), (name, "TEST", tes.shape)

    models[name] = {
        "oof": oof,
        "test": tes,
        "lb": f["lb"],
    }

print("\nLoaded models:")
for k in models:
    print("-", k)

# =========================================================
# 5. REPORT UTILS
# =========================================================
def ba_from_probs(p):
    pred = p.argmax(axis=1)
    return balanced_accuracy_score(y_std, pred)

def recalls_from_probs(p):
    pred = p.argmax(axis=1)
    return recall_score(
        y_std,
        pred,
        labels=[0, 1, 2],
        average=None,
        zero_division=0,
    )

def pred_dist_test(p):
    pred = p.argmax(axis=1)
    return (
        pd.Series([STD_LABELS[i] for i in pred])
        .value_counts(normalize=True)
        .reindex(STD_LABELS)
        .fillna(0.0)
    )

def prob_corr(a, b):
    return np.corrcoef(a.reshape(-1), b.reshape(-1))[0, 1]

def agreement(a, b):
    return (a.argmax(axis=1) == b.argmax(axis=1)).mean()

# =========================================================
# 6. MODEL SUMMARY
# =========================================================
rows = []

for name, m in models.items():
    oof_ba = ba_from_probs(m["oof"])
    rec = recalls_from_probs(m["oof"])
    dist = pred_dist_test(m["test"])

    rows.append({
        "model": name,
        "lb": m["lb"],
        "oof_ba": oof_ba,
        "recall_high": rec[0],
        "recall_low": rec[1],
        "recall_medium": rec[2],
        "test_high": dist["High"],
        "test_low": dist["Low"],
        "test_medium": dist["Medium"],
    })

model_summary = pd.DataFrame(rows).sort_values("lb", ascending=False).reset_index(drop=True)
model_summary.to_csv("model_summary_S_D_A.csv", index=False)

print("\n" + "=" * 90)
print("Model summary")
print("=" * 90)
display(model_summary)

# =========================================================
# 7. CORRELATION / AGREEMENT
# =========================================================
names = list(models.keys())
corr_rows = []

for i in range(len(names)):
    for j in range(i + 1, len(names)):
        n1, n2 = names[i], names[j]
        p1 = models[n1]["oof"]
        p2 = models[n2]["oof"]

        corr_rows.append({
            "m1": n1,
            "m2": n2,
            "prob_corr": prob_corr(p1, p2),
            "agreement": agreement(p1, p2),
            "disagree_oof": int((p1.argmax(axis=1) != p2.argmax(axis=1)).sum()),
        })

corr_df = pd.DataFrame(corr_rows).sort_values("prob_corr").reset_index(drop=True)
corr_df.to_csv("corr_agreement_S_D_A.csv", index=False)

print("\n" + "=" * 90)
print("Correlation / agreement")
print("=" * 90)
display(corr_df)

# =========================================================
# 8. BLEND UTILS
# =========================================================
S_NAME = "S_xgb_seed42_lowmore_0.98111"
D_NAME = "D_xgb_orig_domain_tunedavg_0.98052"
A_NAME = "A_2repro_xgbote_0.98037"

def blend_probs(weights, key="oof"):
    p = np.zeros_like(models[S_NAME][key], dtype=np.float64)

    for name, w in weights.items():
        if w == 0:
            continue
        p += float(w) * models[name][key]

    return normalize_probs(p)

def make_submission_from_std_probs(test_probs, filename):
    pred = test_probs.argmax(axis=1)

    sub = pd.DataFrame({
        "id": test_ids,
        TARGET: [STD_LABELS[i] for i in pred],
    })

    sub.to_csv(filename, index=False)

    dist = (
        sub[TARGET]
        .value_counts(normalize=True)
        .reindex(STD_LABELS)
        .fillna(0.0)
    )

    print("\nSaved:", filename)
    print(dist)
    print(sub.head())

    return sub

def weights_tag(weights):
    parts = []

    mapping = [
        ("S", S_NAME),
        ("D", D_NAME),
        ("A", A_NAME),
    ]

    for short, name in mapping:
        w = float(weights.get(name, 0.0))
        if w > 0:
            parts.append(f"{short}{w:.3f}".replace(".", "p"))

    return "_".join(parts)

def eval_blend(weights, save=False, prefix="submission_blend"):
    oof_p = blend_probs(weights, key="oof")
    test_p = blend_probs(weights, key="test")

    oof_ba = ba_from_probs(oof_p)
    rec = recalls_from_probs(oof_p)
    dist = pred_dist_test(test_p)

    s_pred = models[S_NAME]["test"].argmax(axis=1)
    b_pred = test_p.argmax(axis=1)
    diff_vs_s = int((b_pred != s_pred).sum())

    tag = weights_tag(weights)
    filename = f"{prefix}_{tag}.csv"

    row = {
        "filename": filename,
        "oof_ba": oof_ba,
        "recall_high": rec[0],
        "recall_low": rec[1],
        "recall_medium": rec[2],
        "test_high": dist["High"],
        "test_low": dist["Low"],
        "test_medium": dist["Medium"],
        "diff_vs_S": diff_vs_s,
        "w_S": float(weights.get(S_NAME, 0.0)),
        "w_D": float(weights.get(D_NAME, 0.0)),
        "w_A": float(weights.get(A_NAME, 0.0)),
    }

    if save:
        make_submission_from_std_probs(test_p, filename)

    return row

# =========================================================
# 9. FIXED S-HEAVY BLENDS
# =========================================================
fixed_weight_list = [
    # S only
    {S_NAME: 1.000, D_NAME: 0.000, A_NAME: 0.000},

    # S + D
    {S_NAME: 0.997, D_NAME: 0.003, A_NAME: 0.000},
    {S_NAME: 0.995, D_NAME: 0.005, A_NAME: 0.000},
    {S_NAME: 0.992, D_NAME: 0.008, A_NAME: 0.000},
    {S_NAME: 0.990, D_NAME: 0.010, A_NAME: 0.000},
    {S_NAME: 0.985, D_NAME: 0.015, A_NAME: 0.000},
    {S_NAME: 0.980, D_NAME: 0.020, A_NAME: 0.000},
    {S_NAME: 0.975, D_NAME: 0.025, A_NAME: 0.000},
    {S_NAME: 0.970, D_NAME: 0.030, A_NAME: 0.000},

    # S + A
    {S_NAME: 0.997, D_NAME: 0.000, A_NAME: 0.003},
    {S_NAME: 0.995, D_NAME: 0.000, A_NAME: 0.005},
    {S_NAME: 0.990, D_NAME: 0.000, A_NAME: 0.010},
    {S_NAME: 0.985, D_NAME: 0.000, A_NAME: 0.015},
    {S_NAME: 0.980, D_NAME: 0.000, A_NAME: 0.020},

    # S + D + A
    {S_NAME: 0.995, D_NAME: 0.004, A_NAME: 0.001},
    {S_NAME: 0.992, D_NAME: 0.006, A_NAME: 0.002},
    {S_NAME: 0.990, D_NAME: 0.008, A_NAME: 0.002},
    {S_NAME: 0.985, D_NAME: 0.012, A_NAME: 0.003},
    {S_NAME: 0.980, D_NAME: 0.015, A_NAME: 0.005},
    {S_NAME: 0.975, D_NAME: 0.020, A_NAME: 0.005},
]

fixed_rows = []

for weights in fixed_weight_list:
    row = eval_blend(weights, save=True, prefix="submission_blend")
    fixed_rows.append(row)

fixed_df = pd.DataFrame(fixed_rows)
fixed_df = fixed_df.sort_values("oof_ba", ascending=False).reset_index(drop=True)
fixed_df.to_csv("blend_S_D_A_fixed_summary.csv", index=False)

print("\n" + "=" * 90)
print("Fixed S-heavy blend summary")
print("=" * 90)
display(fixed_df)

# =========================================================
# 10. CONSTRAINED GRID SEARCH
# =========================================================
grid_rows = []

# Sを主軸にする。S >= 0.94だけ探索
for w_D in np.arange(0.000, 0.061, 0.0025):
    for w_A in np.arange(0.000, 0.031, 0.0025):
        w_S = 1.0 - w_D - w_A

        if w_S < 0.94:
            continue

        weights = {
            S_NAME: float(w_S),
            D_NAME: float(w_D),
            A_NAME: float(w_A),
        }

        row = eval_blend(weights, save=False, prefix="submission_grid")
        grid_rows.append(row)

grid_df = pd.DataFrame(grid_rows)
grid_df = grid_df.sort_values("oof_ba", ascending=False).reset_index(drop=True)
grid_df.to_csv("blend_S_D_A_grid_summary.csv", index=False)

print("\n" + "=" * 90)
print("Grid top 30 by OOF")
print("=" * 90)
display(grid_df.head(30))

# =========================================================
# 11. SAVE SELECTED GRID CANDIDATES
# =========================================================
# S単体から数行しか変わらないものはLBもほぼ同じになりやすい。
# ただし変えすぎも危険なので、10〜600行差分の範囲から保存。
selected_grid = grid_df[
    (grid_df["diff_vs_S"] >= 10)
    & (grid_df["diff_vs_S"] <= 600)
].copy()

selected_grid = selected_grid.head(12).reset_index(drop=True)
selected_grid.to_csv("blend_S_D_A_grid_selected.csv", index=False)

print("\n" + "=" * 90)
print("Selected grid candidates to save")
print("=" * 90)
display(selected_grid)

for _, r in selected_grid.iterrows():
    weights = {
        S_NAME: float(r["w_S"]),
        D_NAME: float(r["w_D"]),
        A_NAME: float(r["w_A"]),
    }
    eval_blend(weights, save=True, prefix="submission_grid")

# =========================================================
# 12. FINAL PRINT
# =========================================================
print("\nDone.")
print("\nMain output summaries:")
print("- model_summary_S_D_A.csv")
print("- corr_agreement_S_D_A.csv")
print("- blend_S_D_A_fixed_summary.csv")
print("- blend_S_D_A_grid_summary.csv")
print("- blend_S_D_A_grid_selected.csv")

print("\nFirst submissions to check:")
first_files = [
    "submission_blend_S1p000.csv",
    "submission_blend_S0p997_D0p003.csv",
    "submission_blend_S0p995_D0p005.csv",
    "submission_blend_S0p992_D0p008.csv",
    "submission_blend_S0p990_D0p010.csv",
    "submission_blend_S0p985_D0p015.csv",
    "submission_blend_S0p995_A0p005.csv",
    "submission_blend_S0p990_A0p010.csv",
    "submission_blend_S0p990_D0p008_A0p002.csv",
]

for f in first_files:
    if os.path.exists(f):
        print("-", f)

OOF_ENSEMBLE_DIR : /kaggle/input/datasets/nuwaaaa0000/ps06e04-oof-ensemble
TEST_ENSEMBLE_DIR: /kaggle/input/datasets/nuwaaaa0000/ps06e04-test-ensemble
OOF_DIR          : /kaggle/input/datasets/nuwaaaa0000/ps06e04-oof
TEST_DIR         : /kaggle/input/datasets/nuwaaaa0000/ps06e04-test

Data shapes:
train: (630000, 21)
test : (270000, 20)
y_std: (630000,)

DIR: /kaggle/input/datasets/nuwaaaa0000/ps06e04-oof-ensemble
oof_ensemble_2repro_xgbdomain_0.98036.npy
oof_ensemble_2repro_xgbote_0.98037.npy

DIR: /kaggle/input/datasets/nuwaaaa0000/ps06e04-test-ensemble
test_ensemble_2repro_xgbdomain_0.98036.npy
test_ensemble_2repro_xgbote_0.98037.npy

DIR: /kaggle/input/datasets/nuwaaaa0000/ps06e04-oof
oof_et_ote_0.97462.npy
oof_lgb_ote_0.97943.npy
oof_lgb_repro_0.98005.npy
oof_xgb_2stage_0.97920.npy
oof_xgb_domain_0.97900.npy
oof_xgb_orig_domain_multiseed_tunedavg_std_0.98052.npy
oof_xgb_ote_0.97940.npy
oof_xgb_repro_0.97955.npy
oof_xgb_sudhanshu_seed42_low_more_0p030_std_0.98111.npy

DIR: /kaggle/i

,model,lb,oof_ba,recall_high,recall_low,recall_medium,test_high,test_low,test_medium
0,S_xgb_seed42_lowmore_0.98111,0.98111,0.980147,0.977914,0.994518,0.968010,0.038204,0.590363,0.371433
1,D_xgb_orig_domain_tunedavg_0.98052,0.98052,0.979955,0.976677,0.994918,0.968269,0.038122,0.590667,0.371211
2,A_2repro_xgbote_0.98037,0.98037,0.980174,0.978581,0.995223,0.966717,0.038119,0.590989,0.370893



Correlation / agreement


,m1,m2,prob_corr,agreement,disagree_oof
0,S_xgb_seed42_lowmore_0.98111,A_2repro_xgbote_0.98037,0.997224,0.994429,3510
1,D_xgb_orig_domain_tunedavg_0.98052,A_2repro_xgbote_0.98037,0.997828,0.995279,2974
2,S_xgb_seed42_lowmore_0.98111,D_xgb_orig_domain_tunedavg_0.98052,0.999471,0.997644,1484



Saved: submission_blend_S1p000.csv
Irrigation_Need
High      0.038204
Low       0.590363
Medium    0.371433
Name: proportion, dtype: float64
       id Irrigation_Need
0  630000             Low
1  630001             Low
2  630002             Low
3  630003             Low
4  630004             Low

Saved: submission_blend_S0p997_D0p003.csv
Irrigation_Need
High      0.038204
Low       0.590363
Medium    0.371433
Name: proportion, dtype: float64
       id Irrigation_Need
0  630000             Low
1  630001             Low
2  630002             Low
3  630003             Low
4  630004             Low

Saved: submission_blend_S0p995_D0p005.csv
Irrigation_Need
High      0.038204
Low       0.590363
Medium    0.371433
Name: proportion, dtype: float64
       id Irrigation_Need
0  630000             Low
1  630001             Low
2  630002             Low
3  630003             Low
4  630004             Low

Saved: submission_blend_S0p992_D0p008.csv
Irrigation_Need
High      0.038207
Low       0.59

...


Grid top 30 by OOF


,filename,oof_ba,recall_high,recall_low,recall_medium,test_high,test_low,test_medium,diff_vs_S,w_S,w_D,w_A
0,submission_grid_S0p973_A0p028.csv,0.980165,0.977914,0.994564,0.968018,0.038178,0.590389,0.371433,28,0.9725,0.0000,0.0275
1,submission_grid_S0p980_A0p020.csv,0.980165,0.977914,0.994553,0.968027,0.038181,0.590374,0.371444,21,0.9800,0.0000,0.0200
2,submission_grid_S0p978_A0p022.csv,0.980164,0.977914,0.994556,0.968022,0.038185,0.590381,0.371433,24,0.9775,0.0000,0.0225
3,submission_grid_S0p980_D0p003_A0p018.csv,0.980162,0.977914,0.994550,0.968022,0.038185,0.590378,0.371437,19,0.9800,0.0025,0.0175
4,submission_grid_S0p975_A0p025.csv,0.980162,0.977914,0.994558,0.968014,0.038185,0.590385,0.371430,25,0.9750,0.0000,0.0250
5,submission_grid_S0p963_D0p020_A0p018.csv,0.980162,0.977867,0.994564,0.968056,0.038189,0.590393,0.371419,24,0.9625,0.0200,0.0175
6,submission_grid_S0p965_D0p015_A0p020.csv,0.980161,0.977867,0.994564,0.968052,0.038189,0.590393,0.371419,24,0.9650,0.0150,0.0200
7,submission_grid_S0p985_D0p003_A0p013.csv,0.980158,0.977914,0.994542,0.968018,0.038193,0.590374,0.371433,16,0.9850,0.0025,0.0125
8,submission_grid_S0p985_A0p015.csv,0.980158,0.977914,0.994545,0.968014,0.038189,0.590378,0.371433,18,0.9850,0.0000,0.0150
9,submission_grid_S0p983_D0p003_A0p015.csv,0.980158,0.977914,0.994545,0.968014,0.038189,0.590378,0.371433,18,0.9825,0.0025,0.0150



Selected grid candidates to save


,filename,oof_ba,recall_high,recall_low,recall_medium,test_high,test_low,test_medium,diff_vs_S,w_S,w_D,w_A
0,submission_grid_S0p973_A0p028.csv,0.980165,0.977914,0.994564,0.968018,0.038178,0.590389,0.371433,28,0.9725,0.0000,0.0275
1,submission_grid_S0p980_A0p020.csv,0.980165,0.977914,0.994553,0.968027,0.038181,0.590374,0.371444,21,0.9800,0.0000,0.0200
2,submission_grid_S0p978_A0p022.csv,0.980164,0.977914,0.994556,0.968022,0.038185,0.590381,0.371433,24,0.9775,0.0000,0.0225
3,submission_grid_S0p980_D0p003_A0p018.csv,0.980162,0.977914,0.994550,0.968022,0.038185,0.590378,0.371437,19,0.9800,0.0025,0.0175
4,submission_grid_S0p975_A0p025.csv,0.980162,0.977914,0.994558,0.968014,0.038185,0.590385,0.371430,25,0.9750,0.0000,0.0250
5,submission_grid_S0p963_D0p020_A0p018.csv,0.980162,0.977867,0.994564,0.968056,0.038189,0.590393,0.371419,24,0.9625,0.0200,0.0175
6,submission_grid_S0p965_D0p015_A0p020.csv,0.980161,0.977867,0.994564,0.968052,0.038189,0.590393,0.371419,24,0.9650,0.0150,0.0200
7,submission_grid_S0p985_D0p003_A0p013.csv,0.980158,0.977914,0.994542,0.968018,0.038193,0.590374,0.371433,16,0.9850,0.0025,0.0125
8,submission_grid_S0p985_A0p015.csv,0.980158,0.977914,0.994545,0.968014,0.038189,0.590378,0.371433,18,0.9850,0.0000,0.0150
9,submission_grid_S0p983_D0p003_A0p015.csv,0.980158,0.977914,0.994545,0.968014,0.038189,0.590378,0.371433,18,0.9825,0.0025,0.0150



Saved: submission_grid_S0p973_A0p028.csv
Irrigation_Need
High      0.038178
Low       0.590389
Medium    0.371433
Name: proportion, dtype: float64
       id Irrigation_Need
0  630000             Low
1  630001             Low
2  630002             Low
3  630003             Low
4  630004             Low

Saved: submission_grid_S0p980_A0p020.csv
Irrigation_Need
High      0.038181
Low       0.590374
Medium    0.371444
Name: proportion, dtype: float64
       id Irrigation_Need
0  630000             Low
1  630001             Low
2  630002             Low
3  630003             Low
4  630004             Low

Saved: submission_grid_S0p978_A0p022.csv
Irrigation_Need
High      0.038185
Low       0.590381
Medium    0.371433
Name: proportion, dtype: float64
       id Irrigation_Need
0  630000             Low
1  630001             Low
2  630002             Low
3  630003             Low
4  630004             Low

Saved: submission_grid_S0p980_D0p003_A0p018.csv
Irrigation_Need
High      0.038185
Low 

In [2]:
# =========================================================
# Candidate review for S ensemble
#
# Current anchor:
#   S = xgb_sudhanshu_seed42_low_more_0p030_std_0.98111
#
# All npy class order:
#   [High, Low, Medium]
# =========================================================

import os
import numpy as np
import pandas as pd

from sklearn.metrics import balanced_accuracy_score, recall_score

# =========================================================
# 1. Add all candidate models
# =========================================================
CANDIDATE_FILES = {
    # already loaded, but included for comparison
    "A_2repro_xgbote_0.98037": {
        "oof": "oof_ensemble_2repro_xgbote_0.98037.npy",
        "test": "test_ensemble_2repro_xgbote_0.98037.npy",
        "lb": 0.98037,
    },
    "B_2repro_xgbdomain_0.98036": {
        "oof": "oof_ensemble_2repro_xgbdomain_0.98036.npy",
        "test": "test_ensemble_2repro_xgbdomain_0.98036.npy",
        "lb": 0.98036,
    },
    "D_xgb_orig_domain_tunedavg_0.98052": {
        "oof": "oof_xgb_orig_domain_multiseed_tunedavg_std_0.98052.npy",
        "test": "test_xgb_orig_domain_multiseed_tunedavg_std_0.98052.npy",
        "lb": 0.98052,
    },

    # individual candidates
    "lgb_repro_0.98005": {
        "oof": "oof_lgb_repro_0.98005.npy",
        "test": "test_lgb_repro_0.98005.npy",
        "lb": 0.98005,
    },
    "xgb_repro_0.97955": {
        "oof": "oof_xgb_repro_0.97955.npy",
        "test": "test_xgb_repro_0.97955.npy",
        "lb": 0.97955,
    },
    "lgb_ote_0.97943": {
        "oof": "oof_lgb_ote_0.97943.npy",
        "test": "test_lgb_ote_0.97943.npy",
        "lb": 0.97943,
    },
    "xgb_ote_0.97940": {
        "oof": "oof_xgb_ote_0.97940.npy",
        "test": "test_xgb_ote_0.97940.npy",
        "lb": 0.97940,
    },
    "xgb_domain_0.97900": {
        "oof": "oof_xgb_domain_0.97900.npy",
        "test": "test_xgb_domain_0.97900.npy",
        "lb": 0.97900,
    },
    "xgb_2stage_0.97920": {
        "oof": "oof_xgb_2stage_0.97920.npy",
        "test": "test_xgb_2stage_0.97920.npy",
        "lb": 0.97920,
    },
    "et_ote_0.97462": {
        "oof": "oof_et_ote_0.97462.npy",
        "test": "test_et_ote_0.97462.npy",
        "lb": 0.97462,
    },
}

# =========================================================
# 2. Load candidates
# =========================================================
for name, f in CANDIDATE_FILES.items():
    if name in models:
        continue

    oof = load_oof(f["oof"])
    tes = load_test(f["test"])

    assert oof.shape == (len(train), 3), (name, oof.shape)
    assert tes.shape == (len(test), 3), (name, tes.shape)

    models[name] = {
        "oof": normalize_probs(oof),
        "test": normalize_probs(tes),
        "lb": f["lb"],
    }

print("\nTotal loaded models:", len(models))
for k in models:
    print("-", k)

# =========================================================
# 3. Helper functions
# =========================================================
def error_jaccard(p1, p2):
    e1 = p1.argmax(axis=1) != y_std
    e2 = p2.argmax(axis=1) != y_std

    union = np.logical_or(e1, e2).sum()
    if union == 0:
        return 0.0

    return np.logical_and(e1, e2).sum() / union

def test_diff_vs_s(test_probs):
    return int((test_probs.argmax(axis=1) != models[S_NAME]["test"].argmax(axis=1)).sum())

def summarize_model_vs_s(name):
    p = models[name]["oof"]
    t = models[name]["test"]

    rec = recalls_from_probs(p)
    dist = pred_dist_test(t)

    return {
        "model": name,
        "lb": models[name]["lb"],
        "oof_ba": ba_from_probs(p),
        "recall_high": rec[0],
        "recall_low": rec[1],
        "recall_medium": rec[2],
        "test_high": dist["High"],
        "test_low": dist["Low"],
        "test_medium": dist["Medium"],
        "corr_vs_S": prob_corr(models[S_NAME]["oof"], p),
        "agreement_vs_S": agreement(models[S_NAME]["oof"], p),
        "disagree_oof_vs_S": int(
            (models[S_NAME]["oof"].argmax(axis=1) != p.argmax(axis=1)).sum()
        ),
        "error_jaccard_vs_S": error_jaccard(models[S_NAME]["oof"], p),
        "test_diff_vs_S": test_diff_vs_s(t),
    }

# =========================================================
# 4. Candidate summary
# =========================================================
candidate_names = [n for n in models.keys() if n != S_NAME]

candidate_summary = pd.DataFrame([
    summarize_model_vs_s(n) for n in candidate_names
])

candidate_summary = candidate_summary.sort_values(
    ["corr_vs_S", "oof_ba"],
    ascending=[True, False],
).reset_index(drop=True)

candidate_summary.to_csv("candidate_review_vs_S.csv", index=False)

print("\n" + "=" * 100)
print("Candidate review vs S")
print("=" * 100)
display(candidate_summary)

# =========================================================
# 5. S + one candidate thin blend search
# =========================================================
def blend_two(anchor_name, cand_name, w_cand):
    w_anchor = 1.0 - w_cand

    oof_p = normalize_probs(
        w_anchor * models[anchor_name]["oof"] + w_cand * models[cand_name]["oof"]
    )
    test_p = normalize_probs(
        w_anchor * models[anchor_name]["test"] + w_cand * models[cand_name]["test"]
    )
    return oof_p, test_p

def save_submission_from_probs(test_probs, filename):
    pred = test_probs.argmax(axis=1)

    sub = pd.DataFrame({
        "id": test_ids,
        TARGET: [STD_LABELS[i] for i in pred],
    })
    sub.to_csv(filename, index=False)
    return sub

blend_rows = []

# candidateの単体LB/相関に応じて、まずは薄め中心
WEIGHT_GRID = [
    0.0025, 0.0050, 0.0075,
    0.0100, 0.0125, 0.0150,
    0.0200, 0.0250, 0.0300,
    0.0400, 0.0500,
]

for cand_name in candidate_names:
    for w in WEIGHT_GRID:
        oof_p, test_p = blend_two(S_NAME, cand_name, w)

        rec = recalls_from_probs(oof_p)
        dist = pred_dist_test(test_p)

        s_test_pred = models[S_NAME]["test"].argmax(axis=1)
        b_test_pred = test_p.argmax(axis=1)

        blend_rows.append({
            "candidate": cand_name,
            "filename": f"submission_S_plus_{cand_name}_w{w:.4f}.csv"
                .replace(".", "p")
                .replace("/", "_"),
            "w_S": 1.0 - w,
            "w_candidate": w,
            "candidate_lb": models[cand_name]["lb"],
            "oof_ba": ba_from_probs(oof_p),
            "recall_high": rec[0],
            "recall_low": rec[1],
            "recall_medium": rec[2],
            "test_high": dist["High"],
            "test_low": dist["Low"],
            "test_medium": dist["Medium"],
            "diff_vs_S": int((b_test_pred != s_test_pred).sum()),
            "corr_vs_S": prob_corr(models[S_NAME]["oof"], models[cand_name]["oof"]),
            "agreement_vs_S": agreement(models[S_NAME]["oof"], models[cand_name]["oof"]),
            "error_jaccard_vs_S": error_jaccard(models[S_NAME]["oof"], models[cand_name]["oof"]),
        })

blend_review = pd.DataFrame(blend_rows)

blend_review = blend_review.sort_values(
    ["oof_ba", "diff_vs_S"],
    ascending=[False, True],
).reset_index(drop=True)

blend_review.to_csv("S_plus_candidate_blend_review.csv", index=False)

print("\n" + "=" * 100)
print("S + one candidate blend review: top 50 by OOF")
print("=" * 100)
display(blend_review.head(50))

# =========================================================
# 6. Select practical candidates to save
# =========================================================
# S単体からの差分が少なすぎるものはLBも同じになりやすい。
# ただし差分が多すぎるものは壊れやすい。
selected = blend_review[
    (blend_review["diff_vs_S"] >= 10)
    & (blend_review["diff_vs_S"] <= 120)
].copy()

# candidateが偏りすぎないよう、各候補から上位2つまで
selected = (
    selected
    .sort_values(["oof_ba", "diff_vs_S"], ascending=[False, True])
    .groupby("candidate", as_index=False)
    .head(2)
    .sort_values(["oof_ba", "diff_vs_S"], ascending=[False, True])
    .head(15)
    .reset_index(drop=True)
)

selected.to_csv("S_plus_candidate_selected_to_submit.csv", index=False)

print("\n" + "=" * 100)
print("Selected candidates to save")
print("=" * 100)
display(selected)

for _, r in selected.iterrows():
    cand_name = r["candidate"]
    w = float(r["w_candidate"])

    _, test_p = blend_two(S_NAME, cand_name, w)

    filename = r["filename"]
    save_submission_from_probs(test_p, filename)

    print(
        f"Saved {filename} | "
        f"cand={cand_name} | "
        f"w={w:.4f} | "
        f"OOF={r['oof_ba']:.6f} | "
        f"diff={int(r['diff_vs_S'])}"
    )

print("\nDone.")
print("Saved summaries:")
print("- candidate_review_vs_S.csv")
print("- S_plus_candidate_blend_review.csv")
print("- S_plus_candidate_selected_to_submit.csv")


Loaded OOF
file : oof_ensemble_2repro_xgbdomain_0.98036.npy
path : /kaggle/input/datasets/nuwaaaa0000/ps06e04-oof-ensemble/oof_ensemble_2repro_xgbdomain_0.98036.npy
shape: (630000, 3)

Loaded TEST
file : test_ensemble_2repro_xgbdomain_0.98036.npy
path : /kaggle/input/datasets/nuwaaaa0000/ps06e04-test-ensemble/test_ensemble_2repro_xgbdomain_0.98036.npy
shape: (270000, 3)

Loaded OOF
file : oof_lgb_repro_0.98005.npy
path : /kaggle/input/datasets/nuwaaaa0000/ps06e04-oof/oof_lgb_repro_0.98005.npy
shape: (630000, 3)

Loaded TEST
file : test_lgb_repro_0.98005.npy
path : /kaggle/input/datasets/nuwaaaa0000/ps06e04-test/test_lgb_repro_0.98005.npy
shape: (270000, 3)

Loaded OOF
file : oof_xgb_repro_0.97955.npy
path : /kaggle/input/datasets/nuwaaaa0000/ps06e04-oof/oof_xgb_repro_0.97955.npy
shape: (630000, 3)

Loaded TEST
file : test_xgb_repro_0.97955.npy
path : /kaggle/input/datasets/nuwaaaa0000/ps06e04-test/test_xgb_repro_0.97955.npy
shape: (270000, 3)

Loaded OOF
file : oof_lgb_ote_0.97943.npy

,model,lb,oof_ba,recall_high,recall_low,recall_medium,test_high,test_low,test_medium,corr_vs_S,agreement_vs_S,disagree_oof_vs_S,error_jaccard_vs_S,test_diff_vs_S
0,lgb_repro_0.98005,0.98005,0.006308,0.000000,0.004796,0.014130,0.590985,0.370881,0.038133,-0.496639,0.002668,628319,0.010219,269471
1,xgb_repro_0.97955,0.97955,0.006672,0.000000,0.004979,0.015037,0.590844,0.370644,0.038511,-0.494132,0.003035,628088,0.010112,269369
2,et_ote_0.97462,0.97462,0.975401,0.961207,0.996275,0.968721,0.035685,0.592622,0.371693,0.993329,0.992716,4589,0.623986,1900
3,xgb_domain_0.97900,0.97900,0.979524,0.976201,0.995153,0.967219,0.037596,0.591044,0.371359,0.997059,0.994405,3525,0.703791,1282
4,xgb_2stage_0.97920,0.97920,0.979610,0.976391,0.994718,0.967721,0.037870,0.590548,0.371581,0.997129,0.994237,3631,0.696775,1320
5,B_2repro_xgbdomain_0.98036,0.98036,0.980184,0.978723,0.995196,0.966634,0.038122,0.590978,0.370900,0.997142,0.994327,3574,0.701322,1234
6,A_2repro_xgbote_0.98037,0.98037,0.980174,0.978581,0.995223,0.966717,0.038119,0.590989,0.370893,0.997224,0.994429,3510,0.705482,1210
7,lgb_ote_0.97943,0.97943,0.979164,0.972583,0.995469,0.969440,0.036996,0.591407,0.371596,0.997354,0.994925,3197,0.720800,1156
8,xgb_ote_0.97940,0.97940,0.979251,0.975106,0.995256,0.967391,0.037607,0.591267,0.371126,0.997401,0.994667,3360,0.714940,1093
9,D_xgb_orig_domain_tunedavg_0.98052,0.98052,0.979955,0.976677,0.994918,0.968269,0.038122,0.590667,0.371211,0.999471,0.997644,1484,0.862465,272



S + one candidate blend review: top 50 by OOF


,candidate,filename,w_S,w_candidate,candidate_lb,oof_ba,recall_high,recall_low,recall_medium,test_high,test_low,test_medium,diff_vs_S,corr_vs_S,agreement_vs_S,error_jaccard_vs_S
0,xgb_domain_0.97900,submission_S_plus_xgb_domain_0p97900_w0p0300pcsv,0.9700,0.0300,0.97900,0.980172,0.977914,0.994545,0.968056,0.038159,0.590419,0.371422,41,0.997059,0.994405,0.703791
1,xgb_domain_0.97900,submission_S_plus_xgb_domain_0p97900_w0p0400pcsv,0.9600,0.0400,0.97900,0.980167,0.977867,0.994561,0.968073,0.038133,0.590444,0.371422,55,0.997059,0.994405,0.703791
2,xgb_domain_0.97900,submission_S_plus_xgb_domain_0p97900_w0p0250pcsv,0.9750,0.0250,0.97900,0.980165,0.977914,0.994537,0.968043,0.038163,0.590404,0.371433,34,0.997059,0.994405,0.703791
3,A_2repro_xgbote_0.98037,submission_S_plus_A_2repro_xgbote_0p98037_w0p0...,0.9800,0.0200,0.98037,0.980165,0.977914,0.994553,0.968027,0.038181,0.590374,0.371444,21,0.997224,0.994429,0.705482
4,B_2repro_xgbdomain_0.98036,submission_S_plus_B_2repro_xgbdomain_0p98036_w...,0.9800,0.0200,0.98036,0.980164,0.977914,0.994556,0.968022,0.038181,0.590381,0.371437,23,0.997142,0.994327,0.701322
5,A_2repro_xgbote_0.98037,submission_S_plus_A_2repro_xgbote_0p98037_w0p0...,0.9750,0.0250,0.98037,0.980162,0.977914,0.994558,0.968014,0.038185,0.590385,0.371430,25,0.997224,0.994429,0.705482
6,B_2repro_xgbdomain_0.98036,submission_S_plus_B_2repro_xgbdomain_0p98036_w...,0.9750,0.0250,0.98036,0.980162,0.977914,0.994558,0.968014,0.038181,0.590389,0.371430,27,0.997142,0.994327,0.701322
7,A_2repro_xgbote_0.98037,submission_S_plus_A_2repro_xgbote_0p98037_w0p0...,0.9850,0.0150,0.98037,0.980158,0.977914,0.994545,0.968014,0.038189,0.590378,0.371433,18,0.997224,0.994429,0.705482
8,B_2repro_xgbdomain_0.98036,submission_S_plus_B_2repro_xgbdomain_0p98036_w...,0.9850,0.0150,0.98036,0.980156,0.977914,0.994545,0.968010,0.038189,0.590378,0.371433,18,0.997142,0.994327,0.701322
9,xgb_domain_0.97900,submission_S_plus_xgb_domain_0p97900_w0p0500pcsv,0.9500,0.0500,0.97900,0.980155,0.977819,0.994566,0.968081,0.038122,0.590448,0.371430,65,0.997059,0.994405,0.703791



Selected candidates to save


,candidate,filename,w_S,w_candidate,candidate_lb,oof_ba,recall_high,recall_low,recall_medium,test_high,test_low,test_medium,diff_vs_S,corr_vs_S,agreement_vs_S,error_jaccard_vs_S
0,xgb_domain_0.97900,submission_S_plus_xgb_domain_0p97900_w0p0300pcsv,0.9700,0.0300,0.97900,0.980172,0.977914,0.994545,0.968056,0.038159,0.590419,0.371422,41,0.997059,0.994405,0.703791
1,xgb_domain_0.97900,submission_S_plus_xgb_domain_0p97900_w0p0400pcsv,0.9600,0.0400,0.97900,0.980167,0.977867,0.994561,0.968073,0.038133,0.590444,0.371422,55,0.997059,0.994405,0.703791
2,A_2repro_xgbote_0.98037,submission_S_plus_A_2repro_xgbote_0p98037_w0p0...,0.9800,0.0200,0.98037,0.980165,0.977914,0.994553,0.968027,0.038181,0.590374,0.371444,21,0.997224,0.994429,0.705482
3,B_2repro_xgbdomain_0.98036,submission_S_plus_B_2repro_xgbdomain_0p98036_w...,0.9800,0.0200,0.98036,0.980164,0.977914,0.994556,0.968022,0.038181,0.590381,0.371437,23,0.997142,0.994327,0.701322
4,A_2repro_xgbote_0.98037,submission_S_plus_A_2repro_xgbote_0p98037_w0p0...,0.9750,0.0250,0.98037,0.980162,0.977914,0.994558,0.968014,0.038185,0.590385,0.371430,25,0.997224,0.994429,0.705482
5,B_2repro_xgbdomain_0.98036,submission_S_plus_B_2repro_xgbdomain_0p98036_w...,0.9750,0.0250,0.98036,0.980162,0.977914,0.994558,0.968014,0.038181,0.590389,0.371430,27,0.997142,0.994327,0.701322
6,xgb_ote_0.97940,submission_S_plus_xgb_ote_0p97940_w0p0150pcsv,0.9850,0.0150,0.97940,0.980144,0.977867,0.994531,0.968035,0.038181,0.590389,0.371430,21,0.997401,0.994667,0.714940
7,xgb_ote_0.97940,submission_S_plus_xgb_ote_0p97940_w0p0250pcsv,0.9750,0.0250,0.97940,0.980140,0.977819,0.994537,0.968064,0.038159,0.590415,0.371426,36,0.997401,0.994667,0.714940
8,lgb_ote_0.97943,submission_S_plus_lgb_ote_0p97943_w0p0150pcsv,0.9850,0.0150,0.97943,0.980131,0.977819,0.994539,0.968035,0.038148,0.590393,0.371459,29,0.997354,0.994925,0.720800
9,D_xgb_orig_domain_tunedavg_0.98052,submission_S_plus_D_xgb_orig_domain_tunedavg_0...,0.9500,0.0500,0.98052,0.980131,0.977819,0.994550,0.968022,0.038193,0.590385,0.371422,15,0.999471,0.997644,0.862465


Saved submission_S_plus_xgb_domain_0p97900_w0p0300pcsv | cand=xgb_domain_0.97900 | w=0.0300 | OOF=0.980172 | diff=41
Saved submission_S_plus_xgb_domain_0p97900_w0p0400pcsv | cand=xgb_domain_0.97900 | w=0.0400 | OOF=0.980167 | diff=55
Saved submission_S_plus_A_2repro_xgbote_0p98037_w0p0200pcsv | cand=A_2repro_xgbote_0.98037 | w=0.0200 | OOF=0.980165 | diff=21
Saved submission_S_plus_B_2repro_xgbdomain_0p98036_w0p0200pcsv | cand=B_2repro_xgbdomain_0.98036 | w=0.0200 | OOF=0.980164 | diff=23
Saved submission_S_plus_A_2repro_xgbote_0p98037_w0p0250pcsv | cand=A_2repro_xgbote_0.98037 | w=0.0250 | OOF=0.980162 | diff=25
Saved submission_S_plus_B_2repro_xgbdomain_0p98036_w0p0250pcsv | cand=B_2repro_xgbdomain_0.98036 | w=0.0250 | OOF=0.980162 | diff=27
Saved submission_S_plus_xgb_ote_0p97940_w0p0150pcsv | cand=xgb_ote_0.97940 | w=0.0150 | OOF=0.980144 | diff=21
Saved submission_S_plus_xgb_ote_0p97940_w0p0250pcsv | cand=xgb_ote_0.97940 | w=0.0250 | OOF=0.980140 | diff=36
Saved submission_S_plus_

In [3]:
# =========================================================
# Candidate review with AUTO COLUMN ALIGNMENT
#
# Goal:
#   Fix possible class order issues.
#
# Standard order:
#   [High, Low, Medium]
#
# Some old npy may be:
#   [Low, Medium, High]
# =========================================================

import itertools
import re
import numpy as np
import pandas as pd
from sklearn.metrics import balanced_accuracy_score, recall_score

STD_LABELS = ["High", "Low", "Medium"]
S_NAME = "S_xgb_seed42_lowmore_0.98111"

def normalize_probs(p):
    p = np.asarray(p, dtype=np.float64)
    p = np.clip(p, 1e-15, None)
    p = p / p.sum(axis=1, keepdims=True)
    return p

def align_to_std_by_oof(oof_raw, test_raw, y_std, model_name):
    """
    Find best column permutation by OOF BA.
    Returned arrays are aligned to [High, Low, Medium].
    """
    best_perm = None
    best_ba = -1.0

    for perm in itertools.permutations([0, 1, 2]):
        p = normalize_probs(oof_raw[:, perm])
        ba = balanced_accuracy_score(y_std, p.argmax(axis=1))

        if ba > best_ba:
            best_ba = ba
            best_perm = perm

    oof_aligned = normalize_probs(oof_raw[:, best_perm])
    test_aligned = normalize_probs(test_raw[:, best_perm])

    print(
        f"{model_name:45s} | "
        f"best_perm={best_perm} | "
        f"aligned_oof_ba={best_ba:.6f}"
    )

    return oof_aligned, test_aligned, best_perm, best_ba

def safe_name(s):
    s = s.replace(".", "p")
    s = re.sub(r"[^A-Za-z0-9_]+", "_", s)
    return s.strip("_")

# =========================================================
# 1. Reload all candidates raw, then auto-align
# =========================================================

ALL_MODEL_FILES = {
    "S_xgb_seed42_lowmore_0.98111": {
        "oof":  "oof_xgb_sudhanshu_seed42_low_more_0p030_std_0.98111.npy",
        "test": "test_xgb_sudhanshu_seed42_low_more_0p030_std_0.98111.npy",
        "lb": 0.98111,
    },
    "D_xgb_orig_domain_tunedavg_0.98052": {
        "oof":  "oof_xgb_orig_domain_multiseed_tunedavg_std_0.98052.npy",
        "test": "test_xgb_orig_domain_multiseed_tunedavg_std_0.98052.npy",
        "lb": 0.98052,
    },
    "A_2repro_xgbote_0.98037": {
        "oof":  "oof_ensemble_2repro_xgbote_0.98037.npy",
        "test": "test_ensemble_2repro_xgbote_0.98037.npy",
        "lb": 0.98037,
    },
    "B_2repro_xgbdomain_0.98036": {
        "oof":  "oof_ensemble_2repro_xgbdomain_0.98036.npy",
        "test": "test_ensemble_2repro_xgbdomain_0.98036.npy",
        "lb": 0.98036,
    },
    "lgb_repro_0.98005": {
        "oof":  "oof_lgb_repro_0.98005.npy",
        "test": "test_lgb_repro_0.98005.npy",
        "lb": 0.98005,
    },
    "xgb_repro_0.97955": {
        "oof":  "oof_xgb_repro_0.97955.npy",
        "test": "test_xgb_repro_0.97955.npy",
        "lb": 0.97955,
    },
    "lgb_ote_0.97943": {
        "oof":  "oof_lgb_ote_0.97943.npy",
        "test": "test_lgb_ote_0.97943.npy",
        "lb": 0.97943,
    },
    "xgb_ote_0.97940": {
        "oof":  "oof_xgb_ote_0.97940.npy",
        "test": "test_xgb_ote_0.97940.npy",
        "lb": 0.97940,
    },
    "xgb_domain_0.97900": {
        "oof":  "oof_xgb_domain_0.97900.npy",
        "test": "test_xgb_domain_0.97900.npy",
        "lb": 0.97900,
    },
    "xgb_2stage_0.97920": {
        "oof":  "oof_xgb_2stage_0.97920.npy",
        "test": "test_xgb_2stage_0.97920.npy",
        "lb": 0.97920,
    },
    "et_ote_0.97462": {
        "oof":  "oof_et_ote_0.97462.npy",
        "test": "test_et_ote_0.97462.npy",
        "lb": 0.97462,
    },
}

aligned_models = {}
align_rows = []

print("\nAuto column alignment:")
print("=" * 100)

for name, f in ALL_MODEL_FILES.items():
    oof_raw = np.load(find_oof_file(f["oof"])).astype(np.float64)
    test_raw = np.load(find_test_file(f["test"])).astype(np.float64)

    oof_aligned, test_aligned, perm, aligned_ba = align_to_std_by_oof(
        oof_raw,
        test_raw,
        y_std,
        name,
    )

    aligned_models[name] = {
        "oof": oof_aligned,
        "test": test_aligned,
        "lb": f["lb"],
        "perm": perm,
        "aligned_oof_ba": aligned_ba,
    }

    align_rows.append({
        "model": name,
        "lb": f["lb"],
        "best_perm": perm,
        "aligned_oof_ba": aligned_ba,
    })

align_df = pd.DataFrame(align_rows).sort_values("aligned_oof_ba", ascending=False).reset_index(drop=True)
align_df.to_csv("auto_alignment_summary.csv", index=False)

display(align_df)

# overwrite models with aligned version
models = aligned_models

# =========================================================
# 2. Basic utilities
# =========================================================

def ba_from_probs(p):
    return balanced_accuracy_score(y_std, p.argmax(axis=1))

def recalls_from_probs(p):
    return recall_score(
        y_std,
        p.argmax(axis=1),
        labels=[0, 1, 2],
        average=None,
        zero_division=0,
    )

def pred_dist_test(p):
    pred = p.argmax(axis=1)
    return (
        pd.Series([STD_LABELS[i] for i in pred])
        .value_counts(normalize=True)
        .reindex(STD_LABELS)
        .fillna(0.0)
    )

def prob_corr(a, b):
    return np.corrcoef(a.reshape(-1), b.reshape(-1))[0, 1]

def agreement(a, b):
    return (a.argmax(axis=1) == b.argmax(axis=1)).mean()

def error_jaccard(p1, p2):
    e1 = p1.argmax(axis=1) != y_std
    e2 = p2.argmax(axis=1) != y_std
    union = np.logical_or(e1, e2).sum()
    if union == 0:
        return 0.0
    return np.logical_and(e1, e2).sum() / union

def save_submission_from_probs(test_probs, filename):
    pred = test_probs.argmax(axis=1)
    sub = pd.DataFrame({
        "id": test_ids,
        TARGET: [STD_LABELS[i] for i in pred],
    })
    sub.to_csv(filename, index=False)
    return sub

# =========================================================
# 3. Candidate review vs S after alignment
# =========================================================

def summarize_model_vs_s(name):
    p = models[name]["oof"]
    t = models[name]["test"]

    rec = recalls_from_probs(p)
    dist = pred_dist_test(t)

    return {
        "model": name,
        "lb": models[name]["lb"],
        "oof_ba": ba_from_probs(p),
        "recall_high": rec[0],
        "recall_low": rec[1],
        "recall_medium": rec[2],
        "test_high": dist["High"],
        "test_low": dist["Low"],
        "test_medium": dist["Medium"],
        "corr_vs_S": prob_corr(models[S_NAME]["oof"], p),
        "agreement_vs_S": agreement(models[S_NAME]["oof"], p),
        "disagree_oof_vs_S": int(
            (models[S_NAME]["oof"].argmax(axis=1) != p.argmax(axis=1)).sum()
        ),
        "error_jaccard_vs_S": error_jaccard(models[S_NAME]["oof"], p),
        "test_diff_vs_S": int(
            (models[S_NAME]["test"].argmax(axis=1) != t.argmax(axis=1)).sum()
        ),
        "perm": models[name]["perm"],
    }

candidate_names = [n for n in models.keys() if n != S_NAME]

candidate_summary_aligned = pd.DataFrame([
    summarize_model_vs_s(n) for n in candidate_names
])

candidate_summary_aligned = candidate_summary_aligned.sort_values(
    ["oof_ba", "corr_vs_S"],
    ascending=[False, True],
).reset_index(drop=True)

candidate_summary_aligned.to_csv("candidate_review_vs_S_aligned.csv", index=False)

print("\n" + "=" * 100)
print("Candidate review vs S after alignment")
print("=" * 100)
display(candidate_summary_aligned)

# =========================================================
# 4. S + one candidate thin blend search after alignment
# =========================================================

def blend_two(anchor_name, cand_name, w_cand):
    w_anchor = 1.0 - w_cand

    oof_p = normalize_probs(
        w_anchor * models[anchor_name]["oof"]
        + w_cand * models[cand_name]["oof"]
    )
    test_p = normalize_probs(
        w_anchor * models[anchor_name]["test"]
        + w_cand * models[cand_name]["test"]
    )
    return oof_p, test_p

WEIGHT_GRID = [
    0.0025, 0.0050, 0.0075,
    0.0100, 0.0125, 0.0150,
    0.0200, 0.0250, 0.0300,
    0.0400, 0.0500,
]

blend_rows = []
s_test_pred = models[S_NAME]["test"].argmax(axis=1)

for cand_name in candidate_names:
    for w in WEIGHT_GRID:
        oof_p, test_p = blend_two(S_NAME, cand_name, w)

        rec = recalls_from_probs(oof_p)
        dist = pred_dist_test(test_p)
        b_test_pred = test_p.argmax(axis=1)

        blend_rows.append({
            "candidate": cand_name,
            "filename": f"submission_aligned_S_plus_{safe_name(cand_name)}_w{w:.4f}.csv".replace(".", "p"),
            "w_S": 1.0 - w,
            "w_candidate": w,
            "candidate_lb": models[cand_name]["lb"],
            "oof_ba": ba_from_probs(oof_p),
            "recall_high": rec[0],
            "recall_low": rec[1],
            "recall_medium": rec[2],
            "test_high": dist["High"],
            "test_low": dist["Low"],
            "test_medium": dist["Medium"],
            "diff_vs_S": int((b_test_pred != s_test_pred).sum()),
            "corr_vs_S": prob_corr(models[S_NAME]["oof"], models[cand_name]["oof"]),
            "agreement_vs_S": agreement(models[S_NAME]["oof"], models[cand_name]["oof"]),
            "error_jaccard_vs_S": error_jaccard(models[S_NAME]["oof"], models[cand_name]["oof"]),
            "perm": models[cand_name]["perm"],
        })

blend_review_aligned = pd.DataFrame(blend_rows)
blend_review_aligned = blend_review_aligned.sort_values(
    ["oof_ba", "diff_vs_S"],
    ascending=[False, True],
).reset_index(drop=True)

blend_review_aligned.to_csv("S_plus_candidate_blend_review_aligned.csv", index=False)

print("\n" + "=" * 100)
print("S + one candidate blend review after alignment: top 60 by OOF")
print("=" * 100)
display(blend_review_aligned.head(60))

# =========================================================
# 5. Select candidates to save
# =========================================================

selected_aligned = blend_review_aligned[
    (blend_review_aligned["diff_vs_S"] >= 10)
    & (blend_review_aligned["diff_vs_S"] <= 150)
].copy()

selected_aligned = (
    selected_aligned
    .sort_values(["oof_ba", "diff_vs_S"], ascending=[False, True])
    .groupby("candidate", as_index=False)
    .head(2)
    .sort_values(["oof_ba", "diff_vs_S"], ascending=[False, True])
    .head(18)
    .reset_index(drop=True)
)

selected_aligned.to_csv("S_plus_candidate_selected_aligned_to_submit.csv", index=False)

print("\n" + "=" * 100)
print("Selected aligned candidates to save")
print("=" * 100)
display(selected_aligned)

for _, r in selected_aligned.iterrows():
    cand_name = r["candidate"]
    w = float(r["w_candidate"])

    _, test_p = blend_two(S_NAME, cand_name, w)

    filename = r["filename"]
    save_submission_from_probs(test_p, filename)

    print(
        f"Saved {filename} | "
        f"cand={cand_name} | "
        f"w={w:.4f} | "
        f"OOF={r['oof_ba']:.6f} | "
        f"diff={int(r['diff_vs_S'])} | "
        f"perm={r['perm']}"
    )

print("\nDone.")
print("Saved summaries:")
print("- auto_alignment_summary.csv")
print("- candidate_review_vs_S_aligned.csv")
print("- S_plus_candidate_blend_review_aligned.csv")
print("- S_plus_candidate_selected_aligned_to_submit.csv")


Auto column alignment:
S_xgb_seed42_lowmore_0.98111                  | best_perm=(0, 1, 2) | aligned_oof_ba=0.980147
D_xgb_orig_domain_tunedavg_0.98052            | best_perm=(0, 1, 2) | aligned_oof_ba=0.979955
A_2repro_xgbote_0.98037                       | best_perm=(0, 1, 2) | aligned_oof_ba=0.980174
B_2repro_xgbdomain_0.98036                    | best_perm=(0, 1, 2) | aligned_oof_ba=0.980184
lgb_repro_0.98005                             | best_perm=(2, 0, 1) | aligned_oof_ba=0.980021
xgb_repro_0.97955                             | best_perm=(2, 0, 1) | aligned_oof_ba=0.979888
lgb_ote_0.97943                               | best_perm=(0, 1, 2) | aligned_oof_ba=0.979164
xgb_ote_0.97940                               | best_perm=(0, 1, 2) | aligned_oof_ba=0.979251
xgb_domain_0.97900                            | best_perm=(0, 1, 2) | aligned_oof_ba=0.979524
xgb_2stage_0.97920                            | best_perm=(0, 1, 2) | aligned_oof_ba=0.979610
et_ote_0.97462                      

,model,lb,best_perm,aligned_oof_ba
0,B_2repro_xgbdomain_0.98036,0.98036,"(0, 1, 2)",0.980184
1,A_2repro_xgbote_0.98037,0.98037,"(0, 1, 2)",0.980174
2,S_xgb_seed42_lowmore_0.98111,0.98111,"(0, 1, 2)",0.980147
3,lgb_repro_0.98005,0.98005,"(2, 0, 1)",0.980021
4,D_xgb_orig_domain_tunedavg_0.98052,0.98052,"(0, 1, 2)",0.979955
5,xgb_repro_0.97955,0.97955,"(2, 0, 1)",0.979888
6,xgb_2stage_0.97920,0.97920,"(0, 1, 2)",0.979610
7,xgb_domain_0.97900,0.97900,"(0, 1, 2)",0.979524
8,xgb_ote_0.97940,0.97940,"(0, 1, 2)",0.979251
9,lgb_ote_0.97943,0.97943,"(0, 1, 2)",0.979164



Candidate review vs S after alignment


,model,lb,oof_ba,recall_high,recall_low,recall_medium,test_high,test_low,test_medium,corr_vs_S,agreement_vs_S,disagree_oof_vs_S,error_jaccard_vs_S,test_diff_vs_S,perm
0,B_2repro_xgbdomain_0.98036,0.98036,0.980184,0.978723,0.995196,0.966634,0.038122,0.590978,0.370900,0.997142,0.994327,3574,0.701322,1234,"(0, 1, 2)"
1,A_2repro_xgbote_0.98037,0.98037,0.980174,0.978581,0.995223,0.966717,0.038119,0.590989,0.370893,0.997224,0.994429,3510,0.705482,1210,"(0, 1, 2)"
2,lgb_repro_0.98005,0.98005,0.980021,0.978343,0.995199,0.966521,0.038133,0.590985,0.370881,0.997039,0.994179,3667,0.695092,1245,"(2, 0, 1)"
3,D_xgb_orig_domain_tunedavg_0.98052,0.98052,0.979955,0.976677,0.994918,0.968269,0.038122,0.590667,0.371211,0.999471,0.997644,1484,0.862465,272,"(0, 1, 2)"
4,xgb_repro_0.97955,0.97955,0.979888,0.979009,0.995012,0.965642,0.038511,0.590844,0.370644,0.996654,0.993935,3821,0.687791,1309,"(2, 0, 1)"
5,xgb_2stage_0.97920,0.97920,0.979610,0.976391,0.994718,0.967721,0.037870,0.590548,0.371581,0.997129,0.994237,3631,0.696775,1320,"(0, 1, 2)"
6,xgb_domain_0.97900,0.97900,0.979524,0.976201,0.995153,0.967219,0.037596,0.591044,0.371359,0.997059,0.994405,3525,0.703791,1282,"(0, 1, 2)"
7,xgb_ote_0.97940,0.97940,0.979251,0.975106,0.995256,0.967391,0.037607,0.591267,0.371126,0.997401,0.994667,3360,0.714940,1093,"(0, 1, 2)"
8,lgb_ote_0.97943,0.97943,0.979164,0.972583,0.995469,0.969440,0.036996,0.591407,0.371596,0.997354,0.994925,3197,0.720800,1156,"(0, 1, 2)"
9,et_ote_0.97462,0.97462,0.975401,0.961207,0.996275,0.968721,0.035685,0.592622,0.371693,0.993329,0.992716,4589,0.623986,1900,"(0, 1, 2)"



S + one candidate blend review after alignment: top 60 by OOF


,candidate,filename,w_S,w_candidate,candidate_lb,oof_ba,recall_high,recall_low,recall_medium,test_high,test_low,test_medium,diff_vs_S,corr_vs_S,agreement_vs_S,error_jaccard_vs_S,perm
0,xgb_domain_0.97900,submission_aligned_S_plus_xgb_domain_0p97900_w...,0.9700,0.0300,0.97900,0.980172,0.977914,0.994545,0.968056,0.038159,0.590419,0.371422,41,0.997059,0.994405,0.703791,"(0, 1, 2)"
1,xgb_domain_0.97900,submission_aligned_S_plus_xgb_domain_0p97900_w...,0.9600,0.0400,0.97900,0.980167,0.977867,0.994561,0.968073,0.038133,0.590444,0.371422,55,0.997059,0.994405,0.703791,"(0, 1, 2)"
2,xgb_domain_0.97900,submission_aligned_S_plus_xgb_domain_0p97900_w...,0.9750,0.0250,0.97900,0.980165,0.977914,0.994537,0.968043,0.038163,0.590404,0.371433,34,0.997059,0.994405,0.703791,"(0, 1, 2)"
3,A_2repro_xgbote_0.98037,submission_aligned_S_plus_A_2repro_xgbote_0p98...,0.9800,0.0200,0.98037,0.980165,0.977914,0.994553,0.968027,0.038181,0.590374,0.371444,21,0.997224,0.994429,0.705482,"(0, 1, 2)"
4,xgb_repro_0.97955,submission_aligned_S_plus_xgb_repro_0p97955_w0...,0.9750,0.0250,0.97955,0.980164,0.977914,0.994561,0.968018,0.038181,0.590385,0.371433,26,0.996654,0.993935,0.687791,"(2, 0, 1)"
5,B_2repro_xgbdomain_0.98036,submission_aligned_S_plus_B_2repro_xgbdomain_0...,0.9800,0.0200,0.98036,0.980164,0.977914,0.994556,0.968022,0.038181,0.590381,0.371437,23,0.997142,0.994327,0.701322,"(0, 1, 2)"
6,A_2repro_xgbote_0.98037,submission_aligned_S_plus_A_2repro_xgbote_0p98...,0.9750,0.0250,0.98037,0.980162,0.977914,0.994558,0.968014,0.038185,0.590385,0.371430,25,0.997224,0.994429,0.705482,"(0, 1, 2)"
7,B_2repro_xgbdomain_0.98036,submission_aligned_S_plus_B_2repro_xgbdomain_0...,0.9750,0.0250,0.98036,0.980162,0.977914,0.994558,0.968014,0.038181,0.590389,0.371430,27,0.997142,0.994327,0.701322,"(0, 1, 2)"
8,xgb_repro_0.97955,submission_aligned_S_plus_xgb_repro_0p97955_w0...,0.9800,0.0200,0.97955,0.980158,0.977914,0.994558,0.968002,0.038189,0.590381,0.371430,23,0.996654,0.993935,0.687791,"(2, 0, 1)"
9,A_2repro_xgbote_0.98037,submission_aligned_S_plus_A_2repro_xgbote_0p98...,0.9850,0.0150,0.98037,0.980158,0.977914,0.994545,0.968014,0.038189,0.590378,0.371433,18,0.997224,0.994429,0.705482,"(0, 1, 2)"



Selected aligned candidates to save


,candidate,filename,w_S,w_candidate,candidate_lb,oof_ba,recall_high,recall_low,recall_medium,test_high,test_low,test_medium,diff_vs_S,corr_vs_S,agreement_vs_S,error_jaccard_vs_S,perm
0,xgb_domain_0.97900,submission_aligned_S_plus_xgb_domain_0p97900_w...,0.9700,0.0300,0.97900,0.980172,0.977914,0.994545,0.968056,0.038159,0.590419,0.371422,41,0.997059,0.994405,0.703791,"(0, 1, 2)"
1,xgb_domain_0.97900,submission_aligned_S_plus_xgb_domain_0p97900_w...,0.9600,0.0400,0.97900,0.980167,0.977867,0.994561,0.968073,0.038133,0.590444,0.371422,55,0.997059,0.994405,0.703791,"(0, 1, 2)"
2,A_2repro_xgbote_0.98037,submission_aligned_S_plus_A_2repro_xgbote_0p98...,0.9800,0.0200,0.98037,0.980165,0.977914,0.994553,0.968027,0.038181,0.590374,0.371444,21,0.997224,0.994429,0.705482,"(0, 1, 2)"
3,xgb_repro_0.97955,submission_aligned_S_plus_xgb_repro_0p97955_w0...,0.9750,0.0250,0.97955,0.980164,0.977914,0.994561,0.968018,0.038181,0.590385,0.371433,26,0.996654,0.993935,0.687791,"(2, 0, 1)"
4,B_2repro_xgbdomain_0.98036,submission_aligned_S_plus_B_2repro_xgbdomain_0...,0.9800,0.0200,0.98036,0.980164,0.977914,0.994556,0.968022,0.038181,0.590381,0.371437,23,0.997142,0.994327,0.701322,"(0, 1, 2)"
5,A_2repro_xgbote_0.98037,submission_aligned_S_plus_A_2repro_xgbote_0p98...,0.9750,0.0250,0.98037,0.980162,0.977914,0.994558,0.968014,0.038185,0.590385,0.371430,25,0.997224,0.994429,0.705482,"(0, 1, 2)"
6,B_2repro_xgbdomain_0.98036,submission_aligned_S_plus_B_2repro_xgbdomain_0...,0.9750,0.0250,0.98036,0.980162,0.977914,0.994558,0.968014,0.038181,0.590389,0.371430,27,0.997142,0.994327,0.701322,"(0, 1, 2)"
7,xgb_repro_0.97955,submission_aligned_S_plus_xgb_repro_0p97955_w0...,0.9800,0.0200,0.97955,0.980158,0.977914,0.994558,0.968002,0.038189,0.590381,0.371430,23,0.996654,0.993935,0.687791,"(2, 0, 1)"
8,lgb_repro_0.98005,submission_aligned_S_plus_lgb_repro_0p98005_w0...,0.9800,0.0200,0.98005,0.980148,0.977867,0.994556,0.968022,0.038189,0.590378,0.371433,22,0.997039,0.994179,0.695092,"(2, 0, 1)"
9,lgb_repro_0.98005,submission_aligned_S_plus_lgb_repro_0p98005_w0...,0.9900,0.0100,0.98005,0.980145,0.977867,0.994537,0.968031,0.038196,0.590374,0.371430,15,0.997039,0.994179,0.695092,"(2, 0, 1)"


Saved submission_aligned_S_plus_xgb_domain_0p97900_w0p0300pcsv | cand=xgb_domain_0.97900 | w=0.0300 | OOF=0.980172 | diff=41 | perm=(0, 1, 2)
Saved submission_aligned_S_plus_xgb_domain_0p97900_w0p0400pcsv | cand=xgb_domain_0.97900 | w=0.0400 | OOF=0.980167 | diff=55 | perm=(0, 1, 2)
Saved submission_aligned_S_plus_A_2repro_xgbote_0p98037_w0p0200pcsv | cand=A_2repro_xgbote_0.98037 | w=0.0200 | OOF=0.980165 | diff=21 | perm=(0, 1, 2)
Saved submission_aligned_S_plus_xgb_repro_0p97955_w0p0250pcsv | cand=xgb_repro_0.97955 | w=0.0250 | OOF=0.980164 | diff=26 | perm=(2, 0, 1)
Saved submission_aligned_S_plus_B_2repro_xgbdomain_0p98036_w0p0200pcsv | cand=B_2repro_xgbdomain_0.98036 | w=0.0200 | OOF=0.980164 | diff=23 | perm=(0, 1, 2)
Saved submission_aligned_S_plus_A_2repro_xgbote_0p98037_w0p0250pcsv | cand=A_2repro_xgbote_0.98037 | w=0.0250 | OOF=0.980162 | diff=25 | perm=(0, 1, 2)
Saved submission_aligned_S_plus_B_2repro_xgbdomain_0p98036_w0p0250pcsv | cand=B_2repro_xgbdomain_0.98036 | w=0.025